# **English 문장 생성 트랜스포머 모델**  

📌[실습 05-26] 라이브러리 불러오기

In [ ]:
import numpy as np; import matplotlib.pyplot as plt
import tensorflow as tf
import random; import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, EarlyStopping

In [ ]:
file = "앨리스_1장.txt"
with open(file, "r", encoding="cp949") as f:
    text = f.read()

# 데이터 전처리
text= text.replace("\n", " ")
text = re.sub(r"([?.!,])", r" \1 ", text)

# 토크나이저 정의
tokenizer = Tokenizer(filters='')
tokenizer.fit_on_texts([text])

# 토큰의 수 계산
token_size = len(tokenizer.word_index) + 1  # 0은 패딩용 특수 목적 토큰

text_encoded = tokenizer.texts_to_sequences([text])[0]
index_word={v: k for k, v in tokenizer.word_index.items()} # 디코딩 사전

# 입력과 타깃 시퀀스 생성
seq_length=20

X, y = [], []
for i in range(len(text_encoded) - seq_length):
    X.append(text_encoded[i : i + seq_length])
    y.append(text_encoded[i + 1 : i + seq_length + 1])

X = np.array(X); y = np.array(y)
print("피처 shape=", X.shape); print(X)
print("타깃 shape=", y.shape); print(y)

# 훈련/검증 분할
split = int(len(X)*0.8)
x_train, x_test=X[:split], X[split:]
y_train, y_test=y[:split], y[split:]

# 모델 설정
SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)
embedding_dim=16
model=Sequential()
model.add(Embedding(input_dim=token_size, output_dim=embedding_dim))
model.add(GRU(64, return_sequences=True))
model.add(Dropout(0.3))
model.add(Dense(token_size, activation='softmax'))

# 학습
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=['accuracy'])
history=model.fit(X, y, epochs=25)  # acc 0.6 수준에서 스톱

피처 shape= (1863, 20)
[[ 16   6 217 ...  11 221  60]
 [  6 217   3 ... 221  60   3]
 [217   3  50 ...  60   3 222]
 ...
 [  1 553   1 ...  18 189 560]
 [553   1 554 ... 189 560   5]
 [  1 554   1 ... 560   5  86]]
타깃 shape= (1863, 20)
[[  6 217   3 ... 221  60   3]
 [217   3  50 ...  60   3 222]
 [  3  50  18 ...   3 222 128]
 ...
 [553   1 554 ... 189 560   5]
 [  1 554   1 ... 560   5  86]
 [554   1 555 ...   5  86  10]]
Epoch 1/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.0629 - loss: 5.9341
Epoch 2/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0725 - loss: 5.4639
Epoch 3/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0731 - loss: 5.4029
Epoch 4/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0789 - loss: 5.3043
Epoch 5/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0873 - loss: 5.1086
Epoch 6/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0978 - loss: 4.8822
Epoch 7/25
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.113

In [ ]:
def generate(seed_text, length, temperature=1.0):
    generated_tokens = []

    for _ in range(length):
        # 1. 시드 텍스트 토큰화 및 시퀀스 길이 맞추기
        curr_tokens = tokenizer.texts_to_sequences([seed_text])[0]
        if len(curr_tokens) > seq_length:
            curr_tokens = curr_tokens[-seq_length:]
        pad_tokens = pad_sequences([curr_tokens], maxlen=seq_length, padding='pre')

        # 2. 모델에 입력하여 확률 예측
        probs_all = model.predict(pad_tokens, verbose=0)  # 모든 타임 스텝
        probs = probs_all[0, -1, :]                       # 마지막 타임 스텝의 확률만 추출

        # 3. 반복 패널티 적용 (최근 생성된 20개 토큰의 확률을 2배 축소)
        # 팁: 나눗셈 비율(1.5~2.0)을 직접 조절하여 페널티 강도를 조절할 수 있습니다.
        for token in set(generated_tokens[-20:]):
            probs[token] /= 2

        # 4. Temperature를 적용하여 확률 분포의 뾰족함을 조절
        p_temp = np.log(probs + 1e-10) / temperature
        next_probs = np.exp(p_temp)
        next_probs = next_probs / np.sum(next_probs)

        # 5. 샘플링 및 시드 텍스트 업데이트
        next_index = np.random.choice(len(next_probs), p=next_probs)
        generated_tokens.append(next_index)
        word = index_word.get(next_index)

        # 특수 토큰이나 존재하지 않는 단어를 만나면 종료
        if not word or word == '<pad>':
            break

        seed_text = seed_text + " " + word  # 시드 업데이트

    return seed_text


In [ ]:
import textwrap
SEED= "the rabbit was"
output = generate(seed_text=SEED, length=300, temperature=0.8)
print(textwrap.fill(output, width=60))

the rabbit was all small , or it , and finding it after ,
ma’am , down the rabbit was no idea what latitude was , you
see it , as she was rather low way “drink me , ” and come to
be noticed ? good opportunity for showing off her sister
round it knowledge , in went the hot bottle made of the
house ! ” (when she put to begin . there the belong to say .
down , and eaten up by the bottle was just in another moment
she came upon a heap of sticks and was no , for she found
herself in a moment: she had peeped into one of the country
is , “i hope they’ll remember her began of the house ! ”
(when she had no idea what latitude was very use somewhere
waiting by the bottle was the cat . alice was good practice
to say quite natural); but when she looked up that it was in
her mind in the world she was going to look down , ” poor
alice had not open among the trouble of sticks and looked
that so one ! but i wonder if how late it’s getting ! how
brave either , for , down she came upon a heap of sticks 

In [ ]:
import textwrap
SEED= "Alice was beginning to"
output = generate(seed_text=SEED, length=300, temperature=1.1)
print(textwrap.fill(output, width=60))

Alice was beginning to see she went it , but it was into a
little three-legged table , wondering let i could shut eat ”
down , you bats ? on like the hall , wondering how with the
words to be worth the little golden key in the lock , i
wonder ? ” when alice soon began of the cupboards as she
large unpleasant several things , and my again . at the roof
. ) there wild fall on anything; marked “poison , ” said pop
down a hall , but she had begun to ? if i wonder ? ” when
she went it , pine-apple , tell me late through the pleasure
of making round a hurry . let alice’s or would have went
dark her lessons in the hall , or ? but rather much matter
which way it had saw falling and dry leaves , fancy me the
neck of the door down , you should an such on she great up
slowly oh it’s much matter or not”; it is asking ! ” (which
was too small , but it was empty: she found herself aloud .
“i wonder if ! it’ll only hung catch see: that it bats to
herself , wondering wouldn’t look i your falling down 

### temperature를 조절해서 더 멋지고 창의적인 문장을 생성해 보세요!!
* 시드 텍스트는 원문의 일부를 사용해야 합니다
* 시퀀스 길이를 바꾸어 실험해 보세요.